# Synthetic Student Performance: Regression and Classification

**Mavis Acquah — PUIT23210062**  
**Course:** Information Security

This notebook analyzes a reproducible, synthetic dataset of 500 student observations. Linear regression predicts exam scores from study hours, attendance, practice tests, and sleep; logistic regression predicts pass/fail. The CSV is included alongside this notebook. The score formula and random seed are documented so the generated answers can be reproduced and checked.

## 1. Import libraries

This notebook requires `scikit-learn` and `matplotlib`. If either import is unavailable in your Jupyter environment, install them in a cell with `%pip install scikit-learn matplotlib`, then restart the kernel.

The data is generated with seed `2026`: study hours range from 0–10, attendance from 60–100%, practice tests from 0–20, and sleep from 4–10 hours. Exam score is `10 + 3*study_hours + 0.2*attendance_pct + 1.5*practice_tests + sleep_hours + score_noise`, where noise is drawn from a normal distribution with mean 0 and standard deviation 2. Pass/fail is defined exactly as `exam_score >= 60`. The included `score_noise` column makes the score formula auditable; it is deliberately excluded from the predictors. Standardizing predictors helps logistic regression converge reliably.

In [ ]:
import csv
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_squared_error,
    precision_score,
    recall_score,
    f1_score,
    r2_score,
)
import matplotlib.pyplot as plt


## 2. Generate and split the data

Generate exactly 500 records from the documented formula and save them to the included CSV. The fixed random seeds make the data and 75/25 evaluation split reproducible.

In [ ]:
rng = np.random.default_rng(2026)
n_observations = 500
dataset = {
    "study_hours": rng.uniform(0, 10, n_observations),
    "attendance_pct": rng.uniform(60, 100, n_observations),
    "practice_tests": rng.integers(0, 21, n_observations),
    "sleep_hours": rng.uniform(4, 10, n_observations),
    "score_noise": rng.normal(0, 2, n_observations),
}
dataset["exam_score"] = (
    10
    + 3 * dataset["study_hours"]
    + 0.2 * dataset["attendance_pct"]
    + 1.5 * dataset["practice_tests"]
    + dataset["sleep_hours"]
    + dataset["score_noise"]
)
dataset["passed"] = (dataset["exam_score"] >= 60).astype(int)

columns = [
    "study_hours", "attendance_pct", "practice_tests", "sleep_hours",
    "score_noise", "exam_score", "passed",
]
dataset_path = Path("student_performance_500.csv")
if not dataset_path.exists() and (Path("notebook") / dataset_path).exists():
    dataset_path = Path("notebook") / dataset_path
if not dataset_path.exists():
    raise FileNotFoundError("student_performance_500.csv must be beside the notebook or in the notebook/ folder.")
with dataset_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(columns)
    writer.writerows(zip(*(dataset[column] for column in columns)))

feature_names = ["study_hours", "attendance_pct", "practice_tests", "sleep_hours"]
X = np.column_stack([dataset[name] for name in feature_names])
exam_score = dataset["exam_score"]
passed = dataset["passed"]

X_train, X_test, score_train, score_test = train_test_split(
    X,
    exam_score,
    test_size=0.25,
    random_state=42,
)
X_train_cls, X_test_cls, pass_train, pass_test = train_test_split(
    X,
    passed,
    test_size=0.25,
    random_state=42,
    stratify=passed,
)

print(f"Dataset: {dataset_path} ({n_observations} observations)")
print(f"Features: {', '.join(feature_names)}")
print(f"Pass rule: exam_score >= 60 ({passed.sum()} pass, {(passed == 0).sum()} fail)")
print(f"Split: {len(score_train)} training, {len(score_test)} test observations")

## 3. Train the logistic regression model

Fit the model using only the training data. The test data remains held out for evaluation.

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
model.fit(X_train_cls, pass_train)
print("Logistic regression model trained successfully")

## 4. Evaluate on the test set

Generate predictions for the held-out test data, then review the confusion matrix and per-class metrics for **fail** and **pass**.

In [ ]:
pass_pred = model.predict(X_test_cls)
cm = confusion_matrix(pass_test, pass_pred, labels=[0, 1])

print("Confusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Fail", "Pass"],
)
disp.plot(cmap=plt.cm.Blues)
plt.title("Logistic Regression Confusion Matrix")
plt.tight_layout()
plt.show()

accuracy = accuracy_score(pass_test, pass_pred)
precision = precision_score(pass_test, pass_pred, zero_division=0)
recall = recall_score(pass_test, pass_pred, zero_division=0)
f1 = f1_score(pass_test, pass_pred, zero_division=0)
print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")
print("\nClassification Report:")
print(classification_report(pass_test, pass_pred, target_names=["Fail", "Pass"], zero_division=0))

## 5. Fit and evaluate linear regression

Fit linear regression to predict the continuous exam score. Report MSE, RMSE, and R-squared on held-out data. Residuals are actual minus predicted, and their z-scores indicate how many standard deviations each residual is from the residual mean.

In [ ]:
regression_model = make_pipeline(StandardScaler(), LinearRegression())
regression_model.fit(X_train, score_train)
score_pred = regression_model.predict(X_test)

mse = mean_squared_error(score_test, score_pred)
rmse = mse ** 0.5
r_squared = r2_score(score_test, score_pred)
residuals = score_test - score_pred
residual_std = residuals.std(ddof=1)
if residual_std == 0:
    raise ValueError("Residual standard deviation is zero; residual z-scores are undefined.")
residual_z_scores = (residuals - residuals.mean()) / residual_std

print(f"Linear regression MSE: {mse:.4f}")
print(f"Linear regression RMSE: {rmse:.4f}")
print(f"Linear regression R-squared: {r_squared:.4f}")
print(f"Residual z-score mean: {residual_z_scores.mean():.4f}")
print(f"Residual z-score standard deviation: {residual_z_scores.std(ddof=1):.4f}")
print("\nFirst 10 residual z-scores:")
print(residual_z_scores[:10].round(3))

## 6. Plot 10 held-out observations

The first plot compares actual exam scores with the linear-regression predictions for 10 held-out observations. The second plots their residuals (actual minus predicted); the horizontal zero line marks a perfect prediction.

In [ ]:
n_plot = min(10, len(score_test))
observation_numbers = range(1, n_plot + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(observation_numbers, score_test[:n_plot], "o-", label="Actual")
axes[0].plot(observation_numbers, score_pred[:n_plot], "s--", label="Predicted")
axes[0].set(title="Actual vs. predicted exam scores (10 observations)", xlabel="Test observation", ylabel="Exam score")
axes[0].set_xticks(list(observation_numbers))
axes[0].legend()

axes[1].bar(observation_numbers, residuals[:n_plot], color="steelblue")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(title="Residuals for the same observations", xlabel="Test observation", ylabel="Residual (actual - predicted)")
axes[1].set_xticks(list(observation_numbers))

fig.tight_layout()
plt.show()

## 7. Conclusion

Use the test-set metrics above to summarize performance. The dataset is synthetic and generated by a known formula for educational demonstration; it does not describe real students.